In [1]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system\\Notebook'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system'

In [4]:
from dataclasses import dataclass
from pathlib import Path

In [5]:
@dataclass (frozen=True)
class DataIngestionConfig:
    root_dir : Path
    source_URL : str
    local_data_file : Path
    unzip_dir : Path

In [6]:
from Movie_Recommendation_system.constants import *
from Movie_Recommendation_system.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

In [8]:
import os
import urllib.request as request
import zipfile
from Movie_Recommendation_system import logger
from Movie_Recommendation_system.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")



    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
  

In [16]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-02-08 21:25:55,576: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-08 21:25:55,583: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-08 21:25:55,589: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-02-08 21:25:55,595: INFO: common: created directory at: artifacts]
[2026-02-08 21:25:55,600: INFO: common: created directory at: artifacts/data_ingestion]
[2026-02-08 21:26:05,329: INFO: 2366278714: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 9290399
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "8b815a12ef1759e1e83bec59addefb6be3419556770870d7abff08c10a3b92c8"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 6143:275322:251550:5E0147:6988B20D
Accept-Ranges: bytes
Date: Sun, 